# 03. Побудова ознак, навчання та порівняння моделей ML

**Мета ноутбука:**
1. Завантажити очищений набір (`4 618 780` рядків) із формату Parquet.
2. Перетворити текст відгуку (`review_text`) у розріджену матрицю ознак методом потокового хешування (`HashingVectorizer`, `n_features=2^13`).
3. Поділити повний набір на навчальну (`80%`) та тестову (`20%`) вибірки зі стратифікацією.
4. Виправити дисбаланс масштабів ознак (`app_id` vs текстові ваги) за допомогою `MaxAbsScaler` перед навчанням лінійної моделі.
5. Порівняти три підходи за метриками **ROC-AUC**, **PR-AUC** та **F1-score**:
   - **Модель 0 (Бейзлайн):** `DummyClassifier (most_frequent)`
   - **Модель 1 (Лінійна):** `SGDClassifier (Logistic Regression)` із масштабуванням ознак
   - **Модель 2 (Градієнтний бустинг):** `LightGBM`
6. Побудувати та зберегти графіки **ROC-кривих** і **Матриці помилок** у `report/figures/`.

In [ ]:
import os
import time
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    roc_curve, confusion_matrix, classification_report
)
import lightgbm as lgb

PARQUET_PATH = "../data/steam_reviews_clean.parquet"
FIGURES_DIR = "../report/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

timings = {}
results = {}

### Крок 1. Завантаження Parquet та потокова векторизація тексту (`float32`)

In [ ]:
t0 = time.perf_counter()
df_pq = pl.read_parquet(PARQUET_PATH)
y = df_pq["target"].to_numpy()

num_features = csr_matrix(
    df_pq.select(["app_id", "review_votes", "char_len", "word_count"]).to_numpy(),
    dtype=np.float32
)

# Потокове читання тексту через генератор для економії оперативної пам'яті
text_iter = (row[0] for row in df_pq.select("review_text").iter_rows())
vectorizer = HashingVectorizer(n_features=2**13, alternate_sign=False, dtype=np.float32)
X_text = vectorizer.transform(text_iter)
del df_pq

X = hstack([num_features, X_text], format="csr", dtype=np.float32)
del num_features, X_text

timings["1. Векторизація ознак"] = round(time.perf_counter() - t0, 2)
print(f"Матриця ознак сформована: {X.shape} за {timings['1. Векторизація ознак']} с")

### Крок 2. Стратифікований поділ на train / test (80% / 20%)

In [ ]:
t0 = time.perf_counter()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
del X, y
timings["2. Поділ train/test"] = round(time.perf_counter() - t0, 2)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,} ({timings['2. Поділ train/test']} с)")

### Крок 3. Модель 0 — Просте рішення (`DummyClassifier`)

In [ ]:
t0 = time.perf_counter()
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
y_prob_dummy = dummy.predict_proba(X_test)[:, 1]
timings["3. DummyClassifier"] = round(time.perf_counter() - t0, 2)

results["DummyClassifier"] = {
    "ROC-AUC": roc_auc_score(y_test, y_prob_dummy),
    "PR-AUC": average_precision_score(y_test, y_prob_dummy),
    "F1-score": f1_score(y_test, y_pred_dummy),
    "Time (s)": timings["3. DummyClassifier"]
}
print(results["DummyClassifier"])

### Крок 4. Модель 1 — Логістична регресія (`SGDClassifier` + `MaxAbsScaler`)

*Примітка:* Без масштабування числові ознаки (`app_id` до 10^6) домінують у градієнтному спуску над текстовими ознаками (`[0, 1]`). Додавання `MaxAbsScaler` (який зберігає розрідженість матриці) усуває цю проблему.

In [ ]:
t0 = time.perf_counter()
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sgd = SGDClassifier(loss="log_loss", alpha=1e-5, max_iter=15, n_jobs=-1, random_state=42)
sgd.fit(X_train_scaled, y_train)
y_pred_sgd = sgd.predict(X_test_scaled)
y_prob_sgd = sgd.predict_proba(X_test_scaled)[:, 1]
del X_train_scaled, X_test_scaled
timings["4. LogisticRegression (Scaled)"] = round(time.perf_counter() - t0, 2)

results["LogisticRegression"] = {
    "ROC-AUC": roc_auc_score(y_test, y_prob_sgd),
    "PR-AUC": average_precision_score(y_test, y_prob_sgd),
    "F1-score": f1_score(y_test, y_pred_sgd),
    "Time (s)": timings["4. LogisticRegression (Scaled)"]
}
print(results["LogisticRegression"])

### Крок 5. Модель 2 — Градієнтний бустинг (`LightGBM`)

In [ ]:
t0 = time.perf_counter()
lgbm = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_bin=63,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)
lgbm.fit(X_train, y_train)
y_pred_lgb = lgbm.predict(X_test)
y_prob_lgb = lgbm.predict_proba(X_test)[:, 1]
timings["5. LightGBM"] = round(time.perf_counter() - t0, 2)

results["LightGBM"] = {
    "ROC-AUC": roc_auc_score(y_test, y_prob_lgb),
    "PR-AUC": average_precision_score(y_test, y_prob_lgb),
    "F1-score": f1_score(y_test, y_pred_lgb),
    "Time (s)": timings["5. LightGBM"]
}
print(results["LightGBM"])
print("\nДетальний звіт класифікації LightGBM:")
print(classification_report(y_test, y_pred_lgb, digits=4))

### Крок 6. Порівняльна таблиця та графіки якості моделей

In [ ]:
import pandas as pd
summary_df = pd.DataFrame(results).T
print(summary_df.round(4))

# Графік 4: ROC-криві всіх моделей
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for name, probs, color in [
    ("DummyClassifier (AUC = 0.500)", y_prob_dummy, "#95a5a6"),
    (f"Logistic Regression (AUC = {results['LogisticRegression']['ROC-AUC']:.3f})", y_prob_sgd, "#3498db"),
    (f"LightGBM (AUC = {results['LightGBM']['ROC-AUC']:.3f})", y_prob_lgb, "#2ecc71"),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[0].plot(fpr, tpr, label=name, lw=2, color=color)

axes[0].set_title("ROC-криві моделей на тестовій вибірці", fontsize=12, fontweight="bold")
axes[0].set_xlabel("False Positive Rate (FPR)")
axes[0].set_ylabel("True Positive Rate (TPR)")
axes[0].legend(loc="lower right")

# Графік 5: Матриця помилок для найкращої моделі (LightGBM)
cm = confusion_matrix(y_test, y_pred_lgb)
sns.heatmap(
    cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=axes[1],
    xticklabels=["Негативний (0)", "Позитивний (1)"],
    yticklabels=["Негативний (0)", "Позитивний (1)"]
)
axes[1].set_title("Матриця помилок (LightGBM)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Прогнозований клас")
axes[1].set_ylabel("Справжній клас")

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/04_model_evaluation.png", dpi=300)
plt.show()